# Whole Life Insurance — Liability Cashflow Model

This notebook builds a simple actuarial cashflow projection for a whole life insurance policy:

1. Mortality table setup (placeholder now, swappable for a real SOA 2017 CSO table)
2. Single-policy expected cashflow projection (premiums in, death benefit out)
3. Reserve calculation (PV of net liability cashflows)
4. Aggregation across a book of policies

**Assumptions used:**
- Premiums: paid at start of year, if alive (whole life pay — continues until death)
- Death benefit: paid at end of year of death
- Mortality: 2017 CSO Composite/Unisex, Ultimate table (age-only qx, no select period)


In [9]:
import numpy as np
import pandas as pd


## 1. Mortality Table

Currently using a **placeholder** table (Gompertz-Makeham formula) shaped exactly like the real
2017 CSO CSV (`age`, `qx` columns), so it's a one-line swap once the real file is available.

To use the real table, download it from https://mort.soa.org/ (search "2017 CSO", pick the
Composite/Unisex Ultimate table, export as CSV), then use `load_soa_table()` below instead.


In [10]:
# --- PLACEHOLDER mortality table (Gompertz-Makeham) ---
# Replace with load_soa_table() once the real 2017 CSO CSV is available
ages = np.arange(0, 121)
A, B, c = 0.0002, 0.0000027, 1.1  # illustrative constants
qx_table = np.clip(A + B * c**ages, 0, 1)

mortality_df = pd.DataFrame({"age": ages, "qx": qx_table})
mortality_df.head()


,age,qx
0,0,0.000203
1,1,0.000203
2,2,0.000203
3,3,0.000204
4,4,0.000204


In [11]:
def load_soa_table(filepath, has_duration=False):
    """
    Parses a CSV downloaded from mort.soa.org.
    has_duration=True  -> select & ultimate table (columns: issue_age, duration, qx)
    has_duration=False -> ultimate/attained-age-only table (columns: age, qx)

    NOTE: SOA CSVs typically have no header row. Column layout can vary slightly
    by table type -- inspect the raw file first if this doesn't parse cleanly.
    """
    if has_duration:
        df = pd.read_csv(filepath, header=None, names=["issue_age", "duration", "qx"])
    else:
        df = pd.read_csv(filepath, header=None, names=["age", "qx"])
    return df

# Example usage once you have the real file:
# mortality_df = load_soa_table("2017_CSO_Composite_Ultimate.csv", has_duration=False)


## 2. Single-Policy Cashflow Projection

For one policyholder, projects the **expected** liability cashflow each future year:
- `expected_premium_cf` = probability alive at start of year x annual premium
- `expected_benefit_cf` = probability of death during year x sum assured
- `net_liability_cf` = benefit outflow − premium inflow
- Discounted to get the **reserve** (PV of net liability cashflows at issue)


In [12]:
def project_policy_cashflows(issue_age, sum_assured, annual_premium,
                               mortality_df, max_age=120, interest_rate=0.03):
    """
    Projects expected liability cashflows for a single whole life policy.
    Premiums assumed paid at start of year if alive (whole life pay).
    Death benefit assumed paid at end of year of death.
    """
    qx = mortality_df.set_index("age")["qx"]
    n_years = max_age - issue_age

    tpx = np.zeros(n_years + 1)   # probability alive at start of duration t
    tpx[0] = 1.0
    death_prob = np.zeros(n_years)  # probability death occurs during duration t

    for t in range(n_years):
        age_t = issue_age + t
        q = qx.get(age_t, 1.0)  # force qx=1 at table end
        death_prob[t] = tpx[t] * q
        tpx[t + 1] = tpx[t] * (1 - q)

    duration = np.arange(1, n_years + 1)
    survival_start_of_year = tpx[:-1]     # alive at start -> premium paid
    prob_death_in_year = death_prob        # death during year -> benefit paid

    expected_premium_cf = survival_start_of_year * annual_premium
    expected_benefit_cf = prob_death_in_year * sum_assured
    net_liability_cf = expected_benefit_cf - expected_premium_cf

    discount_factors = 1 / (1 + interest_rate) ** duration

    df = pd.DataFrame({
        "duration": duration,
        "age": issue_age + duration - 1,
        "survival_prob": survival_start_of_year,
        "death_prob": prob_death_in_year,
        "expected_premium_cf": expected_premium_cf,
        "expected_benefit_cf": expected_benefit_cf,
        "net_liability_cf": net_liability_cf,
        "discount_factor": discount_factors,
        "pv_net_cf": net_liability_cf * discount_factors
    })
    return df


In [13]:
# Example: 35yo, $100,000 sum assured, $1,200 annual premium
policy_cf = project_policy_cashflows(
    issue_age=35, sum_assured=100_000, annual_premium=1200,
    mortality_df=mortality_df, interest_rate=0.03
)

policy_cf.head(10)


,duration,age,survival_prob,death_prob,expected_premium_cf,expected_benefit_cf,net_liability_cf,discount_factor,pv_net_cf
0,1,35,1.000000,0.000276,1200.000000,27.587658,-1172.412342,0.970874,-1138.264410
1,2,36,0.999724,0.000283,1199.668948,28.338604,-1171.330344,0.942596,-1104.091191
2,3,37,0.999441,0.000292,1199.328885,29.164746,-1170.164139,0.915142,-1070.865952
3,4,38,0.999149,0.000301,1198.978908,30.073561,-1168.905347,0.888487,-1038.557261
4,5,39,0.998848,0.000311,1198.618025,31.073263,-1167.544762,0.862609,-1007.134368
5,6,40,0.998538,0.000322,1198.245146,32.172881,-1166.072265,0.837484,-976.567164
6,7,41,0.998216,0.000334,1197.859071,33.382335,-1164.476737,0.813092,-946.826150
7,8,42,0.997882,0.000347,1197.458483,34.712524,-1162.745959,0.789409,-917.882397
8,9,43,0.997535,0.000362,1197.041933,36.175424,-1160.866509,0.766417,-889.707517
9,10,44,0.997173,0.000378,1196.607828,37.784189,-1158.823639,0.744094,-862.273618


In [14]:
print(f"Reserve (PV of net liability outflow) at issue: {policy_cf['pv_net_cf'].sum():,.2f}")


Reserve (PV of net liability outflow) at issue: -20,424.15


## 3. Aggregation Across a Book of Policies

Once the per-policy engine works, a full book is just running the same function per policy
and summing (or averaging) results across the book -- no change needed to the core logic.


In [15]:
policies = pd.DataFrame([
    {"policy_id": 1, "issue_age": 35, "sum_assured": 100_000, "annual_premium": 1200},
    {"policy_id": 2, "issue_age": 45, "sum_assured": 250_000, "annual_premium": 3400},
    # add more policies here
])

all_cf = []
for _, row in policies.iterrows():
    cf = project_policy_cashflows(row.issue_age, row.sum_assured, row.annual_premium, mortality_df)
    cf["policy_id"] = row.policy_id
    cf["calendar_year"] = cf["duration"]  # assumes all issued in the same year; offset otherwise
    all_cf.append(cf)

book_cf = pd.concat(all_cf, ignore_index=True)
book_cf.head()


,duration,age,survival_prob,death_prob,expected_premium_cf,expected_benefit_cf,net_liability_cf,discount_factor,pv_net_cf,policy_id,calendar_year
0,1,35,1.000000,0.000276,1200.000000,27.587658,-1172.412342,0.970874,-1138.264410,1,1
1,2,36,0.999724,0.000283,1199.668948,28.338604,-1171.330344,0.942596,-1104.091191,1,2
2,3,37,0.999441,0.000292,1199.328885,29.164746,-1170.164139,0.915142,-1070.865952,1,3
3,4,38,0.999149,0.000301,1198.978908,30.073561,-1168.905347,0.888487,-1038.557261,1,4
4,5,39,0.998848,0.000311,1198.618025,31.073263,-1167.544762,0.862609,-1007.134368,1,5


In [16]:
aggregated = book_cf.groupby("calendar_year")[
    ["expected_premium_cf", "expected_benefit_cf", "net_liability_cf", "pv_net_cf"]
].sum()

aggregated.head(10)


,expected_premium_cf,expected_benefit_cf,net_liability_cf,pv_net_cf
calendar_year,,,,
1,4600.000000,126.788734,-4473.211266,-4342.923559
2,4598.319813,132.418472,-4465.901341,-4209.540335
3,4596.564264,138.608985,-4457.955279,-4079.660592
4,4594.725845,145.415732,-4449.310114,-3953.154408
5,4592.796309,152.899602,-4439.896707,-3829.893901
6,4590.766592,161.127436,-4429.639155,-3709.753055
7,4588.626735,170.172592,-4418.454143,-3592.607557
8,4586.365800,180.115565,-4406.250235,-3478.334624
9,4583.971768,191.044660,-4392.927108,-3366.812839


## Next Steps

- Swap in the real 2017 CSO Composite/Unisex Ultimate table via `load_soa_table()`
- Build the **asset side** (e.g. bond portfolio cashflows) so liability and asset cashflows
  can be compared/netted for ALM analysis
- Consider select & ultimate tables, lapse assumptions, or stochastic interest rate scenarios
  as later refinements
